# 04 — Bronze: Blob Charging Sessions (Hourly Load)
**Day 2 | Step 3 — Databricks side**

Reads hourly CSV files from `dataenggdailystorage` source blob and writes
them to Bronze Delta in ADLS Gen2, then creates an internal Databricks Delta table.

**Run order:**
1. Cell 1 — Configure storage auth (ADLS + source blob SAS)
2. Cell 2 — Detect current hour folder path
3. Cell 3 — Read CSV files for that hour from source blob
4. Cell 4 — Write to Bronze Delta in ADLS (partitioned by date + hour)
5. Cell 5 — Register external Delta table in Hive metastore
6. Cell 6 — Create internal (DBFS-backed) Delta table
7. Cell 7 — Verify both tables

**Prerequisites:**
- Key Vault secrets: `source-storage-account`, `source-sas-token`, `adls-account-name`, SP secrets
- Bronze folder `blob/iot_sessions/` exists

## Cell 1 — Configure storage auth (ADLS + source blob SAS)

Configures two storage connections in a single cell:
1. SP OAuth for your ADLS Gen2 (`evdatalakedev`) — used to write Bronze Delta
2. SAS token for source blob (`dataenggdailystorage`) — used to read CSVs

Both use `spark.conf.set()` with different key patterns:
- `fs.azure.account.auth.type.{account}.dfs...` → OAuth for ADLS
- `fs.azure.sas.{container}.{account}.blob...` → SAS for classic blob

In [ ]:
SCOPE = "kv-ev-scope"

storage_account  = dbutils.secrets.get(scope=SCOPE, key="adls-account-name")
sp_client_id     = dbutils.secrets.get(scope=SCOPE, key="sp-client-id")
sp_client_secret = dbutils.secrets.get(scope=SCOPE, key="sp-client-secret")
sp_tenant_id     = dbutils.secrets.get(scope=SCOPE, key="sp-tenant-id")

src_account   = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
src_sas_token = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")
src_container = "source"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", sp_client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", sp_client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{sp_tenant_id}/oauth2/token")

spark.conf.set(
    f"fs.azure.sas.{src_container}.{src_account}.blob.core.windows.net",
    src_sas_token
)

def abfss(container: str, path: str = "") -> str:
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return f"{base}/{path}" if path else base

print(f"ADLS account  : {storage_account}")
print(f"Source account: {src_account}")
print("Both storage connections configured — OK")

## Cell 2 — Detect current hour folder path

Builds the source blob folder path for the current hour.
Format: `realtime/charging_sessions/YYYY/MM/DD/HH/`

You can override `TARGET_HOUR` to load a specific hour instead of the current one.
Set it to `""` to use the current UTC hour automatically.

`dbutils.fs.ls()` verifies the folder exists before we try to read it.

In [ ]:
from datetime import datetime, timezone

TARGET_HOUR = ""  # Override: set to "2026/07/04/06" to load a specific hour. Leave "" for current hour.

if TARGET_HOUR:
    folder_suffix = TARGET_HOUR
else:
    now           = datetime.now(timezone.utc)
    folder_suffix = now.strftime("%Y/%m/%d/%H")

INGESTION_DATE = folder_suffix[:10].replace("/", "-")
INGESTION_HOUR = folder_suffix[11:13]

SOURCE_PATH = f"wasbs://{src_container}@{src_account}.blob.core.windows.net/realtime/charging_sessions/{folder_suffix}/"

print(f"Source path     : {SOURCE_PATH}")
print(f"Ingestion date  : {INGESTION_DATE}")
print(f"Ingestion hour  : {INGESTION_HOUR}")

try:
    files = dbutils.fs.ls(SOURCE_PATH)
    print(f"Files found     : {len(files)}")
    for f in files[:5]:
        print(f"  {f.name}")
except Exception as e:
    print(f"ERROR: {e}")
    print("The folder does not exist. Check TARGET_HOUR value or try a different hour from Cell 2 of 02_read_source_blob.")
    raise

## Cell 3 — Read CSV files for this hour

Reads all CSV files in the hour folder into a Spark DataFrame.

Two extra columns are added at read time:
- `ingestion_date` — the date portion of the folder path (e.g. `2026-07-04`)
- `ingestion_hour` — the hour (e.g. `06`)

These become partition columns in the Delta table so you can query
`WHERE ingestion_date = '2026-07-04' AND ingestion_hour = '06'` efficiently.

In [ ]:
from pyspark.sql.functions import lit

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SOURCE_PATH)
)

df_raw = df_raw \
    .withColumn("ingestion_date", lit(INGESTION_DATE)) \
    .withColumn("ingestion_hour", lit(INGESTION_HOUR))

print(f"Rows read      : {df_raw.count():,}")
print(f"Columns        : {df_raw.columns}")
print()
df_raw.printSchema()
display(df_raw.limit(5))

## Cell 4 — Write to Bronze Delta (ADLS)

Appends the hour's data to the Bronze Delta table at `bronze/blob/iot_sessions/`.

`partitionBy("ingestion_date", "ingestion_hour")` creates separate Parquet folders
per date and hour. This means:
- Each hour's run writes to a new partition — no overlap with prior hours
- Queries filtering by date or hour skip other partitions entirely (much faster)
- Re-running the same hour appends again — Bronze is raw, duplicates are cleaned in Silver

`mode("append")` means existing data is never overwritten.

In [ ]:
BRONZE_SESSIONS_PATH = abfss("bronze", "blob/iot_sessions/")

df_raw.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("ingestion_date", "ingestion_hour") \
    .save(BRONZE_SESSIONS_PATH)

print(f"Written to : {BRONZE_SESSIONS_PATH}")
print(f"Partition  : ingestion_date={INGESTION_DATE}/ingestion_hour={INGESTION_HOUR}")
print()

files = dbutils.fs.ls(f"{BRONZE_SESSIONS_PATH}ingestion_date={INGESTION_DATE}/ingestion_hour={INGESTION_HOUR}/")
print(f"Parquet files written: {len(files)}")

## Cell 5 — Register external Delta table in Hive metastore

Registers `bronze.charging_sessions` as an external table.
Data stays in ADLS — the metastore just holds the schema and location.

`PARTITIONED BY` tells Hive the partition columns so that
`MSCK REPAIR TABLE` and partition pruning work correctly.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze COMMENT 'Bronze layer — raw ingested data'")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.charging_sessions
    USING DELTA
    LOCATION '{BRONZE_SESSIONS_PATH}'
    COMMENT 'Bronze charging sessions — from source blob, hourly load'
""")

count = spark.sql("SELECT COUNT(*) FROM bronze.charging_sessions").collect()[0][0]
print(f"External table bronze.charging_sessions — OK")
print(f"Row count : {count:,}")
print(f"Location  : {BRONZE_SESSIONS_PATH}")

print()
spark.sql("""
    SELECT ingestion_date, ingestion_hour, COUNT(*) as rows
    FROM bronze.charging_sessions
    GROUP BY ingestion_date, ingestion_hour
    ORDER BY ingestion_date, ingestion_hour
""").show(truncate=False)

## Cell 6 — Create internal (DBFS-backed) Delta table

Copies the ADLS data into a Databricks-managed Delta table stored on DBFS.
This is for learning only — to see the difference between external and internal tables.

Key difference: if you drop this table, the data files are deleted too.
If you drop the external table, the ADLS files are untouched.

In [ ]:
df_adls = spark.read.format("delta").load(BRONZE_SESSIONS_PATH)

df_adls.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ingestion_date", "ingestion_hour") \
    .saveAsTable("bronze.charging_sessions_internal")

ext_count = spark.sql("SELECT COUNT(*) FROM bronze.charging_sessions").collect()[0][0]
int_count = spark.sql("SELECT COUNT(*) FROM bronze.charging_sessions_internal").collect()[0][0]

print(f"Internal table bronze.charging_sessions_internal — OK")
print(f"External rows : {ext_count:,}")
print(f"Internal rows : {int_count:,}")
print(f"Match         : {ext_count == int_count}")

## Cell 7 — Verify both tables

Final check across both tables — row counts, schema, partitions, and Delta history.

In [ ]:
print("=== Verification ===")

ext = spark.sql("SELECT COUNT(*) FROM bronze.charging_sessions").collect()[0][0]
int_ = spark.sql("SELECT COUNT(*) FROM bronze.charging_sessions_internal").collect()[0][0]
print(f"  bronze.charging_sessions (external)  : {ext:,} rows")
print(f"  bronze.charging_sessions_internal    : {int_:,} rows")

print("\nPartitions loaded:")
spark.sql("""
    SELECT ingestion_date, ingestion_hour, COUNT(*) as rows
    FROM bronze.charging_sessions
    GROUP BY ingestion_date, ingestion_hour
    ORDER BY ingestion_date, ingestion_hour
""").show(truncate=False)

print("\nSample records:")
display(spark.sql("""
    SELECT session_id, energy_kwh, session_status, ingestion_date, ingestion_hour
    FROM bronze.charging_sessions
    LIMIT 5
"""))

print("\nDelta history:")
spark.sql("DESCRIBE HISTORY bronze.charging_sessions").select(
    "version", "timestamp", "operation"
).show(5, truncate=False)